In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Orders for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA & REFERENCE TABLES
# ==========================================
df_orders_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_orders"
).filter(F.col("_batch_id") == batch_name)

df_bronze_customers = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_customers"
)
df_silver_customers = spark.read.table(
    "ecommerce_lakehouse.silver.silver_customers"
)

# ==========================================
# 3. TRANSFORMATIONS & SCD TYPE 2 LOOKUP
# ==========================================
# Map bronze order customer_id to customer_unique_id using bronze customers table
# Map bronze order customer_id to customer_unique_id using bronze customers table
df_with_unique_id = df_orders_raw.join(
    df_bronze_customers.select("customer_id", "customer_unique_id"),
    on="customer_id",
    how="left",
)

# Clean order attributes and cast timestamps
df_cleaned = df_with_unique_id.select(
    F.trim(F.col("order_id")).alias("order_id"),
    F.col("customer_id").alias("customer_id"),
    F.col("customer_unique_id").alias("customer_unique_id"),
    utils.clean_text_column("order_status").alias("order_status"),
    F.col("order_purchase_timestamp").cast("timestamp").alias("purchase_ts"),
    F.col("order_approved_at").cast("timestamp").alias("approved_ts"),
    F.col("order_delivered_carrier_date")
    .cast("timestamp")
    .alias("delivered_carrier_ts"),
    F.col("order_delivered_customer_date")
    .cast("timestamp")
    .alias("delivered_customer_ts"),
    F.col("order_estimated_delivery_date")
    .cast("timestamp")
    .alias("estimated_delivery_ts"),
).dropDuplicates(["order_id"])

# Direct join with silver_customers using current active records
df_with_customer_key = df_cleaned.join(
    df_silver_customers.filter(F.col("is_current") == True).select(
        "customer_unique_id", "customer_key"
    ),
    on="customer_unique_id",
    how="left",
).select(
    df_cleaned["*"],
    F.coalesce(F.col("customer_key"), F.lit("C_UNKNOWN")).alias("customer_key"),
)
# Compute Row Hash using SHA-256 for change detection / upserts
df_hashed = df_with_customer_key.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("customer_key"), F.lit("")),
            F.coalesce(F.col("order_status"), F.lit("")),
            F.coalesce(F.col("purchase_ts").cast("string"), F.lit("")),
            F.coalesce(F.col("approved_ts").cast("string"), F.lit("")),
            F.coalesce(
                F.col("delivered_carrier_ts").cast("string"), F.lit("")
            ),
            F.coalesce(
                F.col("delivered_customer_ts").cast("string"), F.lit("")
            ),
            F.coalesce(
                F.col("estimated_delivery_ts").cast("string"), F.lit("")
            ),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE INITIALIZATION OR UPSERT (SCD TYPE 1)
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_orders"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_orders...")

  df_transformed = df_hashed.repartition(1).select(
      "order_id",
      "customer_id",
      "customer_key",
      "order_status",
      "purchase_ts",
      "approved_ts",
      "delivered_carrier_ts",
      "delivered_customer_ts",
      "estimated_delivery_ts",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_orders table!")

else:
  print("Table exists. Executing incremental upsert merge...")

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_hashed.alias("source"),
      condition="target.order_id = source.order_id",
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "customer_key": F.col("source.customer_key"),
          "order_status": F.col("source.order_status"),
          "purchase_ts": F.col("source.purchase_ts"),
          "approved_ts": F.col("source.approved_ts"),
          "delivered_carrier_ts": F.col("source.delivered_carrier_ts"),
          "delivered_customer_ts": F.col("source.delivered_customer_ts"),
          "estimated_delivery_ts": F.col("source.estimated_delivery_ts"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "order_id": F.col("source.order_id"),
          "customer_id": F.col("source.customer_id"),
          "customer_key": F.col("source.customer_key"),
          "order_status": F.col("source.order_status"),
          "purchase_ts": F.col("source.purchase_ts"),
          "approved_ts": F.col("source.approved_ts"),
          "delivered_carrier_ts": F.col("source.delivered_carrier_ts"),
          "delivered_customer_ts": F.col("source.delivered_customer_ts"),
          "estimated_delivery_ts": F.col("source.estimated_delivery_ts"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Orders processing completed successfully!")

Processing Silver Orders for Batch: batch_1
Target table does not exist. Initializing silver_orders...
Successfully initialized silver_orders table!
